# 📦 Section 1 — Import Libraries
> This section loads every external library the notebook needs before any data work begins.
> Importing at the top is best practice: it makes dependencies visible at a glance and avoids *NameError* surprises later.

In [ ]:
# ── Numerical & Data Manipulation ──────────────────────────────────────────
import numpy as np          # NumPy: core library for fast numerical arrays and math operations
import pandas as pd          # Pandas: the main tool for loading, inspecting, and cleaning tabular data

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt  # Matplotlib: low-level plotting engine; used for figure/axis control
import seaborn as sns             # Seaborn: high-level statistical charts built on top of Matplotlib

# ── Standard Library ─────────────────────────────────────────────────────────
import os                   # os: interact with the file system (check paths, list dirs, etc.)

# ── Pandas Extras ────────────────────────────────────────────────────────────
from pandas.plotting import scatter_matrix  # Draws a grid of scatter-plots for all numeric columns at once

# ── Scikit-learn: Pre-processing ─────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder   # Converts text categories to integers (e.g. 'Cat'→0, 'Dog'→1)
from sklearn.preprocessing import MinMaxScaler   # Rescales numbers to [0, 1] so features have equal weight

# ── Scikit-learn: Model Selection ────────────────────────────────────────────
from sklearn.model_selection import train_test_split  # Splits dataset into training and test sets

# ── Scikit-learn: Regression Models ──────────────────────────────────────────
from sklearn.linear_model import LinearRegression        # Simple straight-line regression (baseline model)
from sklearn.tree import DecisionTreeRegressor           # Tree-based regressor; splits data on feature thresholds
from sklearn.ensemble import BaggingRegressor            # Trains many trees on random subsets and averages them (reduces variance)
from sklearn.ensemble import AdaBoostRegressor           # Sequentially corrects errors of weak learners (boosts accuracy)
from sklearn.ensemble import RandomForestRegressor       # Many decision trees with random feature selection (robust & popular)

# ── Scikit-learn: Evaluation Metrics ─────────────────────────────────────────
from sklearn.metrics import r2_score                              # R² score: how much variance the model explains (1.0 = perfect)
from sklearn.metrics import (mean_absolute_error,                # MAE: average absolute error in original units
                              mean_absolute_percentage_error,    # MAPE: error as a % of the true value
                              mean_squared_error)                 # MSE: penalises large errors more heavily than MAE


# 📂 Section 2 — Import Data
> We load the raw CSV file into a Pandas DataFrame. A DataFrame is a two-dimensional
> table with labelled rows (index) and columns — think of it as an Excel sheet in Python.
> `df.head()` previews the first 5 rows so we can quickly verify the data loaded correctly.

In [ ]:
df = pd.read_csv('dirty_cafe_sales.csv')  # Read the CSV file from disk into a DataFrame called 'df'
df.head()                                 # Display the first 5 rows — a quick sanity-check that columns and values look right


# 🔍 Section 3 — Explore the Data
> Before cleaning anything we must *understand* the dataset: how many rows/columns,
> what data types are present, and where the missing or invalid values are.
> This phase is called **Exploratory Data Analysis (EDA)**.

In [ ]:
df.describe()
# describe() returns summary statistics for every numeric column:
# count  → how many non-null values exist
# mean   → arithmetic average
# std    → standard deviation (spread around the mean)
# min/max → smallest and largest value
# 25%/50%/75% → quartiles (25th, 50th/median, 75th percentile)
# Tip: compare 'count' to the total rows to spot missing values at a glance.


In [ ]:
df.info()
# info() prints a concise summary of the DataFrame:
# • Total number of rows (RangeIndex)
# • Each column name, its non-null count, and its inferred dtype
# • dtypes 'object' means text/string; 'float64'/'int64' means numeric
# • If a numeric column shows 'object' here, it contains non-numeric characters — needs cleaning.


In [ ]:
df.isnull().sum()
# isnull() returns a boolean DataFrame: True where a value is NaN/missing
# .sum() counts the True values per column
# Result: a series showing how many NaN values each column has
# This is the first step to understanding the scale of the missing-data problem.


In [ ]:
invalid_values = ["UNKNOWN", "ERROR"]       # List of strings that are not real values — they represent bad/missing data
df.replace(invalid_values, np.nan, inplace=True)  # Replace those strings with NaN (Not a Number) — Pandas' standard missing-value marker
# inplace=True modifies df directly instead of returning a new copy
# After this, every cell that was 'UNKNOWN' or 'ERROR' is now treated as missing.

df.isnull().sum()  # Re-check null counts — the numbers should be higher now that fake strings are converted to NaN


In [ ]:
cols = ["Quantity", "Price Per Unit", "Total Spent"]  # Columns that should be numbers but may contain text garbage

for col in cols:                                        # Loop over each of the three columns
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Try to convert each value to a float
    # errors='coerce' → if conversion fails (e.g. value is a letter), insert NaN instead of raising an error
    # This safely turns any remaining non-numeric strings into NaN so arithmetic works later.


In [ ]:
df.info()
# After the numeric conversion, 'Quantity', 'Price Per Unit', and 'Total Spent'
# should now show float64 dtype instead of object.
# Non-null counts will be lower, reflecting the values that couldn't be converted.


In [ ]:
null_ratio = df.isnull().mean() * 100   # isnull().mean() = fraction of NaN per column; multiply by 100 for percentage
null_ratio.sort_values(ascending=False)  # Sort from most-missing to least-missing so worst columns appear first
# Use this to decide strategy:
# >50% missing → consider dropping the column entirely
# 1-50% missing → impute (fill) with median, mode, or group-based logic


# 🧹 Section 4 — Data Cleaning
> Data cleaning is the process of fixing or removing incorrect, incomplete, or inconsistent data.
> Each step below targets a specific problem found in the exploration phase.
> **Strategy used:** preserve as many rows as possible by imputing (filling) before dropping.

In [ ]:
before = df.shape[0]             # shape[0] = number of rows; save it so we can report how many were deleted

df = df.dropna(subset=['Item'])  # Drop rows where 'Item' (the product name) is NaN
# Without knowing what was sold, the row is useless — we can't impute a product name.
# subset=['Item'] limits the drop to rows missing only in this column.

after = df.shape[0]              # Row count after dropping

print(f"Deleted rows: {before - after}")   # How many rows were removed
print(f"Remaining rows: {after}")          # How many rows survive


In [ ]:
print(df['Price Per Unit'].isnull().sum())  # Check how many Price Per Unit values are still missing
# We'll fill these in the next cell using the median per item — no need to drop them.


In [ ]:
df.isnull().sum()  # Full null audit before the imputation steps below


In [ ]:
df['Price Per Unit'] = df.groupby('Item')['Price Per Unit'].transform(
    lambda x: x.fillna(x.median())  # For each item group, fill NaN with that item's median price
)
# WHY MEDIAN? The median is robust to extreme prices (outliers) that could skew the mean.
# groupby('Item') ensures a 'Coffee' NaN is filled with the median Coffee price, not the
# median of all items — this is called 'group-based imputation' and is much more accurate.


In [ ]:
# ── Derive missing Quantity from the formula: Total Spent = Quantity × Price Per Unit ──
condition = (
    df['Quantity'].isna() &           # Quantity is missing
    df['Total Spent'].notna() &       # but Total Spent is available
    df['Price Per Unit'].notna() &    # and Price Per Unit is available
    (df['Price Per Unit'] != 0)       # and Price is not zero (avoid division by zero)
)

df.loc[condition, 'Quantity'] = (
    df.loc[condition, 'Total Spent'] / df.loc[condition, 'Price Per Unit']
)  # Rearranging: Quantity = Total Spent / Price Per Unit
# df.loc[condition, col] selects only the rows that satisfy 'condition' in that column.
# This is a targeted assignment — rows that already have Quantity are untouched.


In [ ]:
df['Quantity'] = df['Quantity'].astype('Int64')
# Convert Quantity to nullable integer type 'Int64' (capital I)
# Why not regular int64? Regular int64 cannot hold NaN; 'Int64' can.
# Quantities must be whole numbers (you can't sell 1.5 coffees).


In [ ]:
check = df['Quantity'] * df['Price Per Unit']         # Recalculate expected Total Spent

difference = (df['Total Spent'] - check).abs()        # Absolute difference between recorded and expected

print(difference.describe())
# This is a DATA INTEGRITY CHECK:
# If cleaning was correct, most differences should be 0 or very small (rounding only).
# Large differences would signal rows where the three numeric columns are inconsistent.


In [ ]:
df.isnull().sum()  # Mid-cleaning null audit — see how many nulls remain after Quantity imputation


In [ ]:
condition = (
    df['Price Per Unit'].isna() &   # Price is missing
    df['Total Spent'].notna() &     # but Total Spent exists
    df['Quantity'].notna()          # and Quantity exists
)

print(condition.sum())  # Count how many rows could have Price derived: Price = Total Spent / Quantity
# If this is 0, there are no recoverable Price Per Unit values this way.


In [ ]:
# ── Derive missing Total Spent: Total Spent = Quantity × Price Per Unit ──
condition = (
    df['Total Spent'].isna() &       # Total Spent is missing
    df['Quantity'].notna() &         # but Quantity is known
    df['Price Per Unit'].notna()     # and Price Per Unit is known
)

df.loc[condition, 'Total Spent'] = (
    df.loc[condition, 'Quantity'] * df.loc[condition, 'Price Per Unit']
)  # Total Spent = Quantity × Price — reconstruct the value instead of losing the row.


In [ ]:
print(df['Total Spent'].isnull().sum())  # How many Total Spent values are still missing after derivation?


In [ ]:
df = df.dropna(subset=['Quantity', 'Total Spent'])
# Drop rows where BOTH imputation strategies failed:
# • We couldn't derive Quantity (neither Total Spent nor Price was available)
# • We couldn't derive Total Spent (neither Quantity nor Price was available)
# These rows have too little data to be useful for any model.


In [ ]:
df.info()
# After all numeric cleaning, verify:
# • Quantity, Price Per Unit, Total Spent show reduced null counts
# • dtypes are correct (Int64, float64)


In [ ]:
df['Location'] = df.groupby('Item')['Location'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
# Fill missing Location values using the MODE (most frequent value) per Item group.
# mode()[0] → takes the most common Location for that item (e.g. most 'Espresso' sales → 'In-Store')
# 'if not x.mode().empty else x' → safety guard: if a group has no non-null Location at all,
#   leave it as-is rather than crashing. This handles edge cases cleanly.


In [ ]:
print(df['Location'].isnull().sum())  # Confirm: how many Location nulls remain after group-mode imputation?


In [ ]:
df['Payment Method'] = df.groupby('Location')['Payment Method'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
# Fill missing Payment Method using the most common payment method at that Location.
# Rationale: payment preferences differ by location (e.g. 'Takeaway' → mostly Cash).
# Using Location context makes the imputation smarter than a global mode.


In [ ]:
print(df['Payment Method'].isnull().sum())  # Confirm Payment Method nulls are resolved


In [ ]:

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
# Parse the date strings into proper datetime objects.
# errors='coerce' → unparseable dates become NaT (Not a Time) instead of crashing.

# ── Extract temporal features from the date ────────────────────────────────
df['DayOfWeek']  = df['Transaction Date'].dt.dayofweek  # 0=Monday … 6=Sunday
df['Month']      = df['Transaction Date'].dt.month       # 1=January … 12=December
df['Day']        = df['Transaction Date'].dt.day         # Day of month (1–31)
df['Year']       = df['Transaction Date'].dt.year        # 4-digit year
df['YearMonth']  = df['Transaction Date'].dt.to_period('M')  # Period like '2024-03' — useful for monthly aggregations

df['IsWeekend']  = df['DayOfWeek'].isin([5, 6]).astype(int)  # 1 if Saturday or Sunday, else 0 — weekend flag for the model

# ── Fill NaT-derived NaN values ──────────────────────────────────────────────
df['DayOfWeek'].fillna(df['DayOfWeek'].mode()[0], inplace=True)  # Use most common day
df['Month'].fillna(df['Month'].mode()[0], inplace=True)          # Use most common month
df['Day'].fillna(df['Day'].median(), inplace=True)               # Use median day-of-month
df['Year'].fillna(df['Year'].mode()[0], inplace=True)            # Use most common year
df['YearMonth'].fillna(df['YearMonth'].mode()[0], inplace=True)  # Use most common month period
df['IsWeekend'].fillna(0, inplace=True)                          # Default to weekday (0) when unknown

df.drop(columns=['Transaction Date'], inplace=True)  # Drop the original date column — all its information now lives in the new features


In [ ]:
print(df.isnull().sum())  # Final null check after all cleaning steps — should be zero or very close


In [ ]:
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]  # The three numeric columns to inspect for outliers

for col in num_cols:                    # Loop over each column
    plt.figure(figsize=(6,4))           # Create a new figure 6 inches wide, 4 inches tall
    sns.boxplot(x=df[col])             # Draw a box-plot:
    # ┌─────────────────────────────────────────────────────────┐
    # │  Box = IQR (25th–75th percentile), middle line = median │
    # │  Whiskers extend to 1.5×IQR beyond the box             │
    # │  Dots beyond whiskers = potential outliers              │
    # └─────────────────────────────────────────────────────────┘
    plt.title(col)                      # Column name as chart title
    plt.show()                          # Render and display the figure


In [ ]:
def detect_outliers(col):
    Q1 = df[col].quantile(0.25)  # First quartile — 25% of values are below this
    Q3 = df[col].quantile(0.75)  # Third quartile — 75% of values are below this
    IQR = Q3 - Q1                # Interquartile Range: the middle 50% spread

    lower = Q1 - 1.5 * IQR      # Lower fence — values below this are outliers (Tukey's rule)
    upper = Q3 + 1.5 * IQR      # Upper fence — values above this are outliers

    outliers = df[(df[col] < lower) | (df[col] > upper)]  # Filter rows outside the fences

    print(f"{col}: {len(outliers)} outliers")  # Report count

    return lower, upper  # Return bounds for potential downstream capping/clipping

bounds = {}                           # Dictionary to store (lower, upper) bounds per column

for col in num_cols:                  # Apply the function to each numeric column
    bounds[col] = detect_outliers(col)
# bounds can later be used to Winsorise (cap) extreme values if needed.


In [ ]:
df['Check_Total'] = df['Quantity'] * df['Price Per Unit']  # Recalculate expected Total Spent from scratch

df['Difference'] = abs(df['Total Spent'] - df['Check_Total'])  # Absolute difference per row

print(df['Difference'].describe())
# ── Final data integrity check ───────────────────────────────────────────────
# mean ≈ 0 and max ≈ 0 → all three financial columns are perfectly consistent
# Any large values here indicate rows where the cleaned numbers don't add up.


# 📊 Section 5 — Exploratory Data Analysis (EDA)
> With clean data in hand we now explore patterns and distributions through visualisations.
> Each chart answers a specific business question about the café's sales behaviour.

In [ ]:
print(df.head())       # Preview first 5 rows of cleaned data
print(df.describe())   # Summary statistics on the cleaned DataFrame
print(df.info())       # Column types and null counts post-cleaning


In [ ]:
print(df['Payment Method'].unique())  # List all distinct payment methods
# Useful to spot spelling variations or unexpected categories after cleaning.


In [ ]:
top_items = df['Item'].value_counts().head(10)  # Count occurrences of each item; keep top 10
# value_counts() returns a Series sorted by frequency (most common first)

plt.figure(figsize=(8,5))
sns.barplot(x=top_items.values, y=top_items.index)  # Horizontal bar chart: x = count, y = item name
plt.title('Top Selling Items')   # Chart title
plt.xlabel('Count')              # x-axis label: number of transactions
plt.ylabel('Item')               # y-axis label: product name
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Shows which products are ordered most frequently.
# High-volume items drive customer traffic and may benefit from promotions.


In [ ]:
revenue_item = df.groupby('Item')['Total Spent'].sum().sort_values(ascending=False)
# groupby('Item') → group rows by product name
# ['Total Spent'].sum() → sum all sales revenue for each product
# sort_values(ascending=False) → highest revenue first

plt.figure(figsize=(8,5))
sns.barplot(x=revenue_item.values[:10], y=revenue_item.index[:10])  # Top 10 by revenue
plt.title('Revenue by Item')
plt.xlabel('Revenue')
plt.ylabel('Item')
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# A product can be popular (many orders) but low-revenue if it's cheap.
# Comparing this chart to 'Top Selling Items' reveals high-margin vs high-volume products.


In [ ]:
daily_revenue = df.groupby('DayOfWeek')['Total Spent'].sum()
# groupby('DayOfWeek') → group by day number (0=Mon … 6=Sun)
# sum() → total revenue per day

plt.figure(figsize=(8,5))
sns.barplot(x=daily_revenue.index, y=daily_revenue.values)  # x=day number, y=total revenue
plt.title('Revenue by Day of Week')
plt.xlabel('Day (0=Mon, 6=Sun)')
plt.ylabel('Revenue')
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Identifies the busiest (and slowest) days of the week.
# Useful for staffing decisions and targeted marketing on low days.


In [ ]:
location_counts = df['Location'].value_counts()  # Count transactions per location

plt.figure(figsize=(6,4))
sns.barplot(x=location_counts.index, y=location_counts.values)  # Location name vs count
plt.title('Location Distribution')
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Shows whether sales are evenly spread across locations or concentrated in one.
# Imbalanced locations could mean one branch is much busier (or the data is biased).


In [ ]:
payment_counts = df['Payment Method'].value_counts()  # Count transactions per payment type

plt.figure(figsize=(6,4))
sns.barplot(x=payment_counts.index, y=payment_counts.values)
plt.title('Payment Methods')
plt.xticks(rotation=45)  # Rotate x-axis labels 45° so they don't overlap
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Reveals customer payment preferences (Cash vs Card vs Digital).
# Informs decisions about POS systems and digital payment promotion.


In [ ]:
loc_rev = df.groupby('Location')['Total Spent'].mean()  # Average (mean) revenue per transaction per location

sns.barplot(x=loc_rev.index, y=loc_rev.values)
plt.title('Average Revenue by Location')
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Average spend per transaction differs from total revenue:
# a location with fewer transactions but higher average spend is a high-value branch.
# This chart separates volume from value.


In [ ]:
sns.histplot(df['Quantity'], kde=True)  # Histogram with a KDE (Kernel Density Estimate) curve overlaid
# Histogram: divides the value range into bins and counts how many rows fall in each bin
# KDE: smooth continuous curve estimating the probability distribution of the data
plt.title('Quantity Distribution')
plt.show()
# ── What this chart tells us ─────────────────────────────────────────────────
# Shows the shape of the Quantity distribution:
# • Right-skewed → most orders are small, a few are very large
# • Symmetric/normal → orders are evenly spread around a central value
# • Helps decide whether to apply log-transformation before modelling.


# ⚙️ Section 6 — Feature Engineering
> Feature engineering creates new columns (features) that are more informative for
> a machine-learning model than the raw columns alone.
> Good features compress domain knowledge into signals the model can learn from.

In [ ]:
id = "drop_id"                                    # Variable stored for documentation; not used in logic
df.drop(columns=['Transaction ID'], inplace=True)  # Remove the ID column
# Transaction ID is a unique identifier with no predictive value.
# Including it would cause the model to 'memorise' training IDs instead of learning patterns.


In [ ]:
monthly_df = df.groupby('YearMonth')['Total Spent'].sum().reset_index()
# groupby('YearMonth') → group all transactions by calendar month
# .sum() → total revenue per month
# .reset_index() → converts the grouped result back to a regular DataFrame with a numeric index
monthly_df.columns = ['YearMonth', 'Revenue']  # Rename columns for clarity

monthly_df['YearMonth'] = monthly_df['YearMonth'].dt.to_timestamp()  # Convert Period ('2024-03') to a proper datetime (2024-03-01)
# to_timestamp() is needed before we can use .dt.month, .dt.year on the column.


In [ ]:
monthly_df['Month'] = monthly_df['YearMonth'].dt.month  # Extract month number (1–12) as a numeric feature
monthly_df['Year']  = monthly_df['YearMonth'].dt.year   # Extract year (e.g. 2024)

# ── Lag Features ──────────────────────────────────────────────────────────────
monthly_df['Lag_1'] = monthly_df['Revenue'].shift(1)  # Revenue from 1 month ago
monthly_df['Lag_2'] = monthly_df['Revenue'].shift(2)  # Revenue from 2 months ago
monthly_df['Lag_3'] = monthly_df['Revenue'].shift(3)  # Revenue from 3 months ago
# Lag features let the model see recent history — essential for time-series forecasting.
# shift(n) moves values down n rows, so row i gets the value from row i-n.

# ── Rolling Statistics ────────────────────────────────────────────────────────
monthly_df['Rolling_Mean_3'] = monthly_df['Revenue'].rolling(3).mean()  # Average of last 3 months (smooths noise)
monthly_df['Rolling_Std_3']  = monthly_df['Revenue'].rolling(3).std()   # Std dev of last 3 months (captures volatility)
# rolling(3) creates a sliding window of 3 rows; .mean()/.std() compute stats over that window.

# ── Growth Rate ───────────────────────────────────────────────────────────────
monthly_df['Growth'] = monthly_df['Revenue'].pct_change()  # Month-over-month percentage change
# pct_change() = (current - previous) / previous — measures growth momentum.

monthly_df = monthly_df.dropna()  # Drop the first few rows that have NaN from shift/rolling (not enough history)


In [ ]:
df['High_Spending'] = (
    df['Total Spent'] > df['Total Spent'].median()  # True if this transaction is above the median spend
).astype(int)  # Convert True/False to 1/0 — this creates a binary classification target
# Use case: train a classifier to predict whether a transaction will be high-value.
# Using median as threshold gives a balanced split (~50% each class).


In [ ]:
item_counts = df['Item'].value_counts()   # Count how many times each item appears in the dataset

df['Item_Popularity'] = df['Item'].map(item_counts)  # Map each row's item to its frequency count
# .map() does a dictionary-style lookup: 'Coffee' → 423, 'Tea' → 310, etc.
# This converts a categorical column into a numeric feature the model can use directly.
# Popular items may have different spending patterns from niche items.


In [ ]:
monthly_df.head(10)  # Preview the first 10 rows of the monthly time-series DataFrame
# Should show: YearMonth, Revenue, Month, Year, Lag_1, Lag_2, Lag_3, Rolling_Mean_3, Rolling_Std_3, Growth


In [ ]:
df.info()  # Final overview of the fully cleaned and feature-engineered transaction DataFrame
# Check: all expected columns present, dtypes are correct, null count is zero


# 💾 Section 7 — Export Cleaned Data
> Save the cleaned and feature-engineered DataFrame to a new CSV file.
> This file is the input for downstream modelling notebooks.

In [ ]:
df.to_csv("dirty_cafe.csv", index=False)  # Write the DataFrame to CSV
# index=False → do NOT write the row numbers (0,1,2,...) as a column in the file
# The output file 'dirty_cafe.csv' contains only the meaningful columns.

print("CSV exported successfully!")  # Confirmation message
